# 03 · Operational monitoring

What a data engineer checks each morning: did the runs succeed, what changed upstream,
is quality holding, and is anything waiting for a human (identity review, schema drift)?

In [ ]:
%pip install -q duckdb deltalake

In [ ]:
import duckdb
from deltalake import DeltaTable
ROOT = "/lakehouse/default/Tables"
con = duckdb.connect()
for t in ["pipeline_runs", "dq_results", "ingestion_state", "schema_drift", "bronze_manifest"]:
    try:
        con.register(t, DeltaTable(f"{ROOT}/meta/{t}").to_pyarrow_dataset())
    except Exception as e:
        print("missing", t, e)

In [ ]:
con.sql("""SELECT run_id, step, status, round(duration_s,1) secs, rows_out, left(error,80) error
           FROM pipeline_runs ORDER BY started_at DESC LIMIT 20""").show()

In [ ]:
con.sql("""WITH last AS (SELECT max(run_id) run_id FROM dq_results)
           SELECT rule_id, severity, passed, failed_count FROM dq_results JOIN last USING (run_id)
           ORDER BY passed, severity""").show()

In [ ]:
con.sql("""SELECT competition_id, season_id, count(*) matches, max(last_updated) newest_source_change,
                  count(*) FILTER (WHERE silver_processed_at < bronze_ingested_at OR silver_processed_at IS NULL) pending
           FROM ingestion_state GROUP BY ALL""").show()
con.sql("SELECT json_path, sum(occurrences) n, min(detected_at) first_seen FROM schema_drift GROUP BY ALL").show()